# Nerfstudio auf eigenen Bildern — Colab

Linearer Workflow: **Setup → Bilder hochladen → COLMAP-Posen → splatfacto trainieren → Viewer**.

## Bedienung

- **Zelle für Zelle** ausführen (nicht *Run all*).
- Nach Zelle 2 (`condacolab.install()`) **startet die Runtime automatisch neu**. Das ist gewollt. Danach mit Zelle 3 weitermachen — Zelle 2 nicht erneut.
- Laufzeittyp: **GPU** (Menü → Laufzeit → Laufzeittyp ändern → T4/L4/A100).

## Daten

20–80 überlappende Fotos einer Szene. Gleiche Belichtung, keine bewegten Objekte. Format: JPG/PNG.

## 1. GPU prüfen

In [ ]:
!nvidia-smi

## 2. condacolab installieren

Installiert Miniforge und ersetzt Colabs Default-Python. **Runtime restartet automatisch** — das ist normal. Nach dem Restart mit Zelle 3 weitermachen.

In [ ]:
!pip install -q condacolab
import condacolab
condacolab.install()

## 3. Python 3.10 pinnen

Erwartete Ausgabe am Ende: `Python 3.10.x`.

In [ ]:
import condacolab
condacolab.check()
!mamba install -y python=3.10
!python --version

## 4. CUDA 11.8 + PyTorch 2.0.1

Erwartung am Ende: `torch 2.0.1+cu118 cuda True`. Wenn `cuda False`: Laufzeittyp auf GPU stellen und Notebook neu starten.

In [ ]:
%%bash
set -e
mamba install -y -c "nvidia/label/cuda-11.8.0" cuda-toolkit
pip install torch==2.0.1+cu118 torchvision==0.15.2+cu118 torchaudio==2.0.2+cu118 \
  --extra-index-url https://download.pytorch.org/whl/cu118
python -c "import torch; print('torch', torch.__version__, 'cuda', torch.cuda.is_available())"

## 5. tiny-cuda-nn (vorgebautes Wheel)

Erwartung: `tcnn ok`.

In [ ]:
%%bash
set -e
pip install -q gdown
gdown "https://drive.google.com/u/1/uc?id=1-7x7qQfB7bIw2zV4Lr6-yhvMpjXC84Q5&confirm=t" -O /tmp/tcnn.whl
pip install /tmp/tcnn.whl
python -c "import tinycudann as tcnn; print('tcnn ok')"

## 6. Nerfstudio + COLMAP

Nerfstudio gepinnt auf `1.0.3` (letzte Version mit torch 2.0.1). Erwartung: Liste der `ns-train`-Methoden.

In [ ]:
%%bash
set -e
apt-get install -y -qq colmap ffmpeg
pip install nerfstudio==1.0.3
ns-train --help 2>&1 | head -25

## 7. Bilder hochladen

Wähle deine Fotos. Sie landen in `/content/data/images/`. Die Zelle akzeptiert Mehrfachauswahl.

In [ ]:
import os, shutil
from google.colab import files

IMG_DIR = '/content/data/images'
os.makedirs(IMG_DIR, exist_ok=True)
uploaded = files.upload()
for name in uploaded:
    shutil.move(name, os.path.join(IMG_DIR, name))
print(f'Hochgeladen: {len(uploaded)} Bilder in {IMG_DIR}')

## 8. COLMAP-Posen berechnen

Dauer: ~5–15 Min bei 20–80 Bildern. Ergebnis liegt in `/content/data/processed/`.

In [ ]:
!ns-process-data images \
  --data /content/data/images \
  --output-dir /content/data/processed \
  --matching-method exhaustive

## 9. ngrok-Tunnel für den Viewer (optional, aber empfohlen)

Der Nerfstudio-Viewer läuft auf Port 7007 *in* der Colab-VM — von außen nicht erreichbar. ngrok macht ihn öffentlich.

1. Account bei https://dashboard.ngrok.com/signup (gratis)
2. Token unter https://dashboard.ngrok.com/get-started/your-authtoken kopieren
3. Hier einsetzen und ausführen — die Zelle gibt dir eine URL aus, die du **nach** dem Trainingsstart in Zelle 10 öffnest.

In [ ]:
NGROK_TOKEN = ''  # <- hier dein Token einsetzen

!pip install -q pyngrok
from pyngrok import ngrok, conf
if NGROK_TOKEN:
    conf.get_default().auth_token = NGROK_TOKEN
ngrok.kill()
tunnel = ngrok.connect(7007, 'http')
print('Viewer-URL (sobald Training läuft):', tunnel.public_url)

## 10. Training starten — splatfacto (3D Gaussian Splatting)

`splatfacto` ist die 3DGS-Variante in Nerfstudio (schnell, gute Qualität, viewer-friendly).

Standard: 30000 Iterationen. Bei einer T4 dauert das ~30–60 Min. Für einen schnellen Test runter auf 7000 setzen (`--max-num-iterations 7000`).

In [ ]:
!ns-train splatfacto \
  --data /content/data/processed \
  --output-dir /content/outputs \
  --viewer.websocket-port 7007 \
  --max-num-iterations 30000

## 11. Resultate herunterladen

Die trainierte Szene + Checkpoints liegen unter `/content/outputs/`. Die folgende Zelle zippt alles und lädt es runter.

In [ ]:
import shutil
from google.colab import files
shutil.make_archive('/content/outputs', 'zip', '/content/outputs')
files.download('/content/outputs.zip')